# Extracción de Aranceles (Sección 301 - China)
## Fases de Tratado
### Fase 0: Configuración General

El objetivo de este script es procesar el documento oficial en PDF que contiene los listados de fracciones arancelarias castigadas bajo la Sección 301 (China Tariffs). Utiliza expresiones regulares para rastrear y vincular los códigos HTS regulares con sus respectivos códigos sancionadores del Capítulo 99.

Dependencias requeridas:
- `pdfplumber:` Motor de extracción de texto para iterar sobre las páginas del PDF.
- `pandas (pd):` Manipulación de estructuras de datos y exportación tabular.
- `re:` Motor de expresiones regulares para coincidencia de patrones.
- `os:` Manejo de rutas y creación de directorios.

Variables Globales:
- **Rutas:** Apuntan al archivo Raw (PDF) y definen la ruta de salida procesada (Excel).
- **Patrón Regex:** Define la regla de búsqueda exacta para atrapar las duplas de códigos arancelarios.

In [1]:
import pdfplumber
import pandas as pd
import re
import os

# Rutas de archivos
PATH_PDF_CHINA = '../data/raw/China Tariffs_2025HTSRev32.pdf'
PATH_SALIDA_CHINA = '../data/intermediate/Extraccion_China_Tariffs.xlsx'

# Expresión Regular Central
# Explicación del patrón:
# (\d{4}\.\d{2}\.\d{2})  -> Busca el código HTS estándar (ej: 0101.21.00)
# \s+                    -> Busca cualquier espacio en medio (salto de línea o tabulación)
# (9903\.\d{2}\.\d{2})   -> Busca el código sancionador del Cap 99 (ej: 9903.88.15)
PATRON_REGEX_CHINA = re.compile(r'(\d{4}\.\d{2}\.\d{2})\s+.*?(9903\.\d{2}\.\d{2})')

print("--- CONFIGURACIÓN CARGADA ---")
print(f"Input PDF: {PATH_PDF_CHINA}")
print(f"Output:    {PATH_SALIDA_CHINA}")

### Fase 0.5: Definición de Funciones

Se definen las funciones principales para aislar la lógica de negocio y facilitar su mantenimiento:

**1. Funciones Principales (`nombre`)**
Orquestadores del proceso de lectura y extracción.

- **`extraer_aranceles_china`**: Función encargada de abrir el documento PDF, iterar a través de todas sus páginas extrayendo el texto crudo, y aplicar el patrón de expresión regular para capturar las tuplas de información. Finaliza su proceso eliminando duplicados y retornando un DataFrame estructurado.

In [2]:
# --- FUNCIONES PRINCIPALES ---

def extraer_aranceles_china(pdf_path, patron_regex):
    """
    Abre el PDF de tarifas, itera por cada página y extrae mediante
    expresiones regulares los pares de códigos HTS y su tarifa del Capítulo 99.
    """
    print(f">> Iniciando extracción desde: {pdf_path}...")
    datos_extraidos = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_paginas = len(pdf.pages)
            print(f"   El documento tiene {total_paginas} páginas detectadas.")

            for i, pagina in enumerate(pdf.pages):
                texto = pagina.extract_text()
                
                if texto:
                    coincidencias = patron_regex.findall(texto)
                    for match in coincidencias:
                        hts_code, heading_99 = match
                        datos_extraidos.append({
                            'HTS_Code': hts_code,
                            'Chapter_99_Heading': heading_99
                        })

                # Logger de progreso
                if (i + 1) % 50 == 0:
                    print(f"   Procesando página {i + 1} de {total_paginas}...")

        # Convertir a DataFrame
        df_resultado = pd.DataFrame(datos_extraidos)
        
        # Limpieza básica: eliminar duplicados si el PDF repite encabezados
        df_resultado = df_resultado.drop_duplicates().reset_index(drop=True)
        
        return df_resultado

    except FileNotFoundError:
        print(f"ERROR: No se encontró el archivo PDF en {pdf_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"ERROR CRÍTICO DURANTE LA EXTRACCIÓN: {e}")
        return pd.DataFrame()

### Fase 1: Procesamiento y Extracción (PDF)
Se invoca al orquestador principal pasándole las variables globales configuradas. El sistema comenzará la lectura página por página hasta consolidar toda la información extraída.

In [3]:
DF_CHINA_TARIFFS = extraer_aranceles_china(PATH_PDF_CHINA, PATRON_REGEX_CHINA)
print(f"\n>> Proceso finalizado. Se extrajeron {len(DF_CHINA_TARIFFS)} códigos únicos.")

### Fase 2: Exportación Final
El DataFrame resultante se exporta a formato Excel en la ruta intermedia definida, creando los directorios previos en caso de no existir, y mostrando una vista previa del resultado final.

In [4]:
print(f"Generando Excel: {PATH_SALIDA_CHINA}...")

try:
    if not DF_CHINA_TARIFFS.empty:
        # Crear directorio si no existe
        os.makedirs(os.path.dirname(PATH_SALIDA_CHINA), exist_ok=True)
        
        # Guardado a Excel
        DF_CHINA_TARIFFS.to_excel(PATH_SALIDA_CHINA, index=False)
        
        print("¡ÉXITO! Archivo generado correctamente.")
        print("\nEjemplo de salida:")
        print(DF_CHINA_TARIFFS.head())
    else:
        print("ADVERTENCIA: El DataFrame está vacío, no se exportó ningún archivo.")
except Exception as e:
    print(f"ERROR CRÍTICO EN EXPORTACIÓN: {e}")